# Step3. Delta テーブルの作成

Step1で作成したLakehouseにStep2でアップロードしたJSONLファイルをもとにDelataテーブルを作成します。


このノートブックをStep1で作成したLakehouse **[Prefix]_lh_chemical** にアタッチします。


# 設定
WorkspaceのGUIDおよびLakehouseのGUIDはURLから切り出して下記のセルを書き換えてください。

```
https://app.fabric.microsoft.com/groups/{workspaceId}/lakehouses/{artifactId}?experience.......
```
- groups/ の後ろ → Workspace ID
- lakehouses/ の後ろ → Lakehouse ID（= Artifact ID）?より前の文字列

In [ ]:
# 設定
WORKSPACE_NAME       = "WS_Chemical_HOL"   #👈 Fabric ワークスペースの GUID を入力
LAKEHOUSE_NAME       = "MS_lh_chemical"   #👈 [Prefix]_lh_chemical を入力
REFERENCE_DATA_PATH  = f"Files"

WORKSPACE_ID= "d995c964-321b-4486-9a11-fc23428ef52c"
LAKEHOUSE_ID ="912ed674-bdf6-4240-82ae-dd0556e63a9d"

# スキーマ定義

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, ArrayType, LongType
)

SCHEMAS = {
    "production_lines": StructType([
        StructField("production_line_id", StringType(), False),
        StructField("name", StringType(), False),
        StructField("plant", StringType(), False),
        StructField("process_type", StringType(), False),
        StructField("capacity_tons_per_day", DoubleType(), True),
        StructField("status", StringType(), False),
    ]),
    "equipment": StructType([
        StructField("equipment_id", StringType(), False),
        StructField("equipment_number", StringType(), False),
        StructField("name", StringType(), False),
        StructField("equipment_type", StringType(), False),
        StructField("production_line_id", StringType(), False),
        StructField("installation_year", IntegerType(), True),
        StructField("manufacturer", StringType(), True),
        StructField("status", StringType(), False),
        StructField("last_maintenance_date", StringType(), True),
    ]),
    "sensors": StructType([
        StructField("sensor_id", StringType(), False),
        StructField("tag_name", StringType(), False),
        StructField("measurement_type", StringType(), False),
        StructField("unit", StringType(), False),
        StructField("equipment_id", StringType(), False),
        StructField("normal_min", DoubleType(), True),
        StructField("normal_max", DoubleType(), True),
        StructField("alarm_low", DoubleType(), True),
        StructField("alarm_high", DoubleType(), True),
        StructField("status", StringType(), False),
    ]),
    "products": StructType([
        StructField("product_id", StringType(), False),
        StructField("name", StringType(), False),
        StructField("grade", StringType(), False),
        StructField("spec_lower", DoubleType(), True),
        StructField("spec_upper", DoubleType(), True),
        StructField("unit", StringType(), True),
        StructField("category", StringType(), False),
    ]),
    "process_orders": StructType([
        StructField("process_order_id", StringType(), False),
        StructField("batch_number", StringType(), False),
        StructField("product_id", StringType(), False),
        StructField("production_line_id", StringType(), False),
        StructField("target_quantity", DoubleType(), True),
        StructField("actual_quantity", DoubleType(), True),
        StructField("start_time", StringType(), True),
        StructField("end_time", StringType(), True),
        StructField("status", StringType(), False),
    ]),
    "operation_phases": StructType([
        StructField("operation_phase_id", StringType(), False),
        StructField("process_order_id", StringType(), False),
        StructField("phase_name", StringType(), False),
        StructField("sequence_number", IntegerType(), False),
        StructField("set_temperature", DoubleType(), True),
        StructField("set_pressure", DoubleType(), True),
        StructField("actual_temperature", DoubleType(), True),
        StructField("actual_pressure", DoubleType(), True),
        StructField("primary_sensor_id", StringType(), True),
        StructField("start_time", StringType(), True),
        StructField("end_time", StringType(), True),
        StructField("status", StringType(), False),
    ]),
    "process_deviations": StructType([
        StructField("process_deviation_id", StringType(), False),
        StructField("sensor_id", StringType(), False),
        StructField("operation_phase_id", StringType(), False),
        StructField("deviation_type", StringType(), False),
        StructField("detection_time", StringType(), True),
        StructField("deviation_amount", DoubleType(), True),
        StructField("threshold_value", DoubleType(), True),
        StructField("actual_value", DoubleType(), True),
        StructField("handling_status", StringType(), False),
    ]),
    "quality_results": StructType([
        StructField("quality_result_id", StringType(), False),
        StructField("lot_number", StringType(), False),
        StructField("process_order_id", StringType(), False),
        StructField("product_id", StringType(), False),
        StructField("process_deviation_id", StringType(), True),
        StructField("inspection_item", StringType(), False),
        StructField("measured_value", DoubleType(), True),
        StructField("spec_lower", DoubleType(), True),
        StructField("spec_upper", DoubleType(), True),
        StructField("pass_fail", StringType(), False),
        StructField("inspection_date", StringType(), True),
    ]),
    "failure_events": StructType([
        StructField("failure_event_id", StringType(), False),
        StructField("equipment_id", StringType(), False),
        StructField("process_deviation_id", StringType(), True),
        StructField("occurrence_time", StringType(), True),
        StructField("failure_mode", StringType(), False),
        StructField("impact_scope", StringType(), False),
        StructField("downtime_hours", DoubleType(), True),
        StructField("severity", StringType(), False),
        StructField("resolution_status", StringType(), False),
    ]),
    "root_causes": StructType([
        StructField("root_cause_id", StringType(), False),
        StructField("failure_event_id", StringType(), False),
        StructField("equipment_id", StringType(), False),
        StructField("operation_phase_id", StringType(), True),
        StructField("cause_classification", StringType(), False),
        StructField("description", StringType(), True),
        StructField("corrective_action", StringType(), True),
        StructField("preventive_action", StringType(), True),
        StructField("analysis_date", StringType(), True),
    ]),
}

# Primary key column for each table
PRIMARY_KEYS = {
    "production_lines": "production_line_id",
    "equipment": "equipment_id",
    "sensors": "sensor_id",
    "products": "product_id",
    "process_orders": "process_order_id",
    "operation_phases": "operation_phase_id",
    "process_deviations": "process_deviation_id",
    "quality_results": "quality_result_id",
    "failure_events": "failure_event_id",
    "root_causes": "root_cause_id",
}

# JSONL ファイルをLakehouseテーブルに読み込む

In [ ]:
SCHEMA_NAME = "dbo"   # スキーマ対応 Lakehouse の既定スキーマ

# Order matters: load tables with no FK dependencies first
LOAD_ORDER = [
    "production_lines",
    "equipment",
    "sensors",
    "products",
    "process_orders",
    "operation_phases",
    "process_deviations",
    "quality_results",
    "failure_events",
    "root_causes",
]

results = []

for table_name in LOAD_ORDER:
    file_path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/{REFERENCE_DATA_PATH}/{table_name}.jsonl"
    schema = SCHEMAS[table_name]
    pk = PRIMARY_KEYS[table_name]

    try:
        # Read JSONL (JSON Lines format = one JSON object per line)
        df = spark.read.schema(schema).json(file_path)

        # Verify no null primary keys
        null_pk_count = df.filter(df[pk].isNull()).count()
        if null_pk_count > 0:
            print(f"⚠ WARNING: {table_name} has {null_pk_count} null primary keys ({pk})")

        # Verify primary key uniqueness
        total = df.count()
        distinct_pk = df.select(pk).distinct().count()
        if total != distinct_pk:
            print(f"⚠ WARNING: {table_name} has {total - distinct_pk} duplicate primary keys ({pk})")

        # Write as Delta table (overwrite to support re-runs)
        df.write.format("delta").mode("overwrite").saveAsTable(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{SCHEMA_NAME}.{table_name}")
        #df.write.format("delta").mode("overwrite").saveAsTable(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{table_name}") #Lakehouse スキーマを使用しない場合

        results.append({"table": table_name, "rows": total, "pk": pk, "status": "✓ loaded"})
        print(f"  ✓ {table_name}: {total} rows loaded (PK: {pk})")

    except Exception as e:
        results.append({"table": table_name, "rows": 0, "pk": pk, "status": f"✗ {str(e)[:80]}"})

In [ ]:
# summary display
from pyspark.sql import Row

summary_df = spark.createDataFrame([Row(**r) for r in results])
display(summary_df)

# 検証：行数とサンプルデータ

In [ ]:
print("=" * 60)
print("TABLE ROW COUNTS")
print("=" * 60)
for table_name in LOAD_ORDER:
     try:
         count = spark.table(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{SCHEMA_NAME}.{table_name}").count()
         #count = spark.table(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{table_name}").count() #Lakehouse スキーマを使用しない場合
         print(f"  {table_name:<25} {count:>6} rows")
     except Exception as e:
         print(f"  {table_name:<25} ERROR: {e}")

In [ ]:
 # Display sample records from key tables
for table_name in ["production_lines", "equipment", "sensors", "process_orders"]:
     print(f"\n{'=' * 60}")
     print(f"SAMPLE: {table_name} (first 3 rows)")
     print("=" * 60)
     display(spark.table(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{SCHEMA_NAME}.{table_name}").limit(3))
     #display(spark.table(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{table_name}").limit(3)) #Lakehouseスキーマを使用しない場合